In [ ]:
%pip install pyspark==4.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798675 sha256=66e5be244227794c6049da6d47d989193a78213f9a76f004df87962ee6841a70
  Stored in directory: /root/.cache/pip/wheels/4f/6d/69/496f1f5964e3470c76313fe2a499eb219a9c95ac1f1cb880ab
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.4
    Uninstalling pyspark-4.0.4:
      Successfully uninstalled pyspark-4.0.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.2.0 which is incompatible.


In [ ]:
# creating the spark session
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("spark_adaptive_query_execution")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [ ]:
# creating dataset
from pyspark.sql import functions as F

N = 1_000_000

base_df = (
    spark.range(0, N)
    .withColumn(
        "amount",
        (F.col("id") % 10000).cast("double")
    )
    .withColumn(
        "category",
        F.when(F.col("id") % 10 == 0, "A")
         .when(F.col("id") % 10 == 1, "B")
         .otherwise("C")
    )
    .cache()
)

print("Dataset created.")
print("Expected rows:", N)
print("Actual rows:", base_df.count())

base_df.show(5)
base_df.printSchema()

Dataset created.
Expected rows: 1000000
Actual rows: 1000000
+---+------+--------+
| id|amount|category|
+---+------+--------+
|  0|   0.0|       A|
|  1|   1.0|       B|
|  2|   2.0|       C|
|  3|   3.0|       C|
|  4|   4.0|       C|
+---+------+--------+
only showing top 5 rows
root
 |-- id: long (nullable = false)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = false)



In [ ]:
# regular python udf
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# python function to classify amount into categories
def classify_amount_python(amount):
    if amount >= 7000:
        return "HIGH"
    elif amount >= 3000:
        return "MEDIUM"
    else:
        return "LOW"

# converting python fxns into udf
classify_amount_udf = udf(
    classify_amount_python,
    StringType(),
    useArrow=False
)

# creating a new col in the spark dataframe using the python udf
udf_df = base_df.withColumn(
    "amount_status",
    classify_amount_udf(F.col("amount"))
)


In [ ]:
# Implementation 2: Built-in Spark expression
builtin_df = base_df.withColumn(
    "amount_status",
    F.when(F.col("amount") >= 7000, "HIGH")
     .when(F.col("amount") >= 3000, "MEDIUM")
     .otherwise("LOW")
)



In [ ]:
# Compare the output counts from both implementations
udf_counts = (
    udf_df.groupBy("amount_status")
    .count()
    .withColumnRenamed("count", "row_count")
)

builtin_counts = (
    builtin_df.groupBy("amount_status")
    .count()
    .withColumnRenamed("count", "row_count")
)

In [ ]:
print("Regular Python UDF results:")
udf_counts.orderBy("amount_status").show()

print("Built-in Spark expression results:")
builtin_counts.orderBy("amount_status").show()

Regular Python UDF results:
+-------------+---------+
|amount_status|row_count|
+-------------+---------+
|         HIGH|   300000|
|          LOW|   300000|
|       MEDIUM|   400000|
+-------------+---------+

Built-in Spark expression results:
+-------------+---------+
|amount_status|row_count|
+-------------+---------+
|         HIGH|   300000|
|          LOW|   300000|
|       MEDIUM|   400000|
+-------------+---------+



In [ ]:
# Check whether both implementations produce identical grouped results
udf_results = {
    row["amount_status"]: row["row_count"]
    for row in udf_counts.collect()
}

builtin_results = {
    row["amount_status"]: row["row_count"]
    for row in builtin_counts.collect()
}

assert udf_results == builtin_results, "Results do not match!"

print("VALIDATION PASSED: Both implementations produce identical results.")

VALIDATION PASSED: Both implementations produce identical results.


In [ ]:
print("REGULAR PYTHON UDF — FORMATTED EXECUTION PLAN")
udf_df.explain("formatted")

REGULAR PYTHON UDF — FORMATTED EXECUTION PLAN
== Physical Plan ==
* Project (6)
+- BatchEvalPython (5)
   +- InMemoryTableScan (1)
         +- InMemoryRelation (2)
               +- * Project (4)
                  +- * Range (3)


(1) InMemoryTableScan
Output [3]: [id#170L, amount#171, category#172]
Arguments: [id#170L, amount#171, category#172]

(2) InMemoryRelation
Arguments: [id#170L, amount#171, category#172], StorageLevel(disk, memory, deserialized, 1 replicas), [id#170L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(4) Project [codegen id : 1]
Output [3]: [id#0L, cast((id#0L % 10000) as double) AS amount#1, CASE WHEN ((id#0L % 10) = 0) THEN A WHEN ((id#0L % 10) = 1) THEN B ELSE C END AS category#2]
Input [1]: [id#0L]

(5) BatchEvalPython
Input [3]: [id#170L, amount#171, category#172]
Arguments: [classify_amount_python(amount#171)#325], [pythonUDF0#693]

(6) Project [codegen id : 1]
Output [4]: [id#170L, amo

In [ ]:
print("BUILT-IN SPARK EXPRESSION — FORMATTED EXECUTION PLAN")
builtin_df.explain("formatted")

BUILT-IN SPARK EXPRESSION — FORMATTED EXECUTION PLAN
== Physical Plan ==
* Project (5)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- * Project (4)
               +- * Range (3)


(1) InMemoryTableScan
Output [3]: [amount#171, category#172, id#170L]
Arguments: [amount#171, category#172, id#170L]

(2) InMemoryRelation
Arguments: [id#170L, amount#171, category#172], StorageLevel(disk, memory, deserialized, 1 replicas), [id#170L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(4) Project [codegen id : 1]
Output [3]: [id#0L, cast((id#0L % 10000) as double) AS amount#1, CASE WHEN ((id#0L % 10) = 0) THEN A WHEN ((id#0L % 10) = 1) THEN B ELSE C END AS category#2]
Input [1]: [id#0L]

(5) Project [codegen id : 1]
Output [4]: [id#170L, amount#171, category#172, CASE WHEN (amount#171 >= 7000.0) THEN HIGH WHEN (amount#171 >= 3000.0) THEN MEDIUM ELSE LOW END AS amount_status#328]
Input [3]: [amount#171, c

In [ ]:
# fxn to measure the time taken to execute the action

import time
import statistics

def benchmark_dataframe(df, label):
    start_time = time.perf_counter()

    results = (
        df.groupBy("amount_status")
        .count()
        .collect()
    )

    elapsed_seconds = time.perf_counter() - start_time

    print(f"{label}: {elapsed_seconds:.3f} seconds")
    print("Result:", results)

    return elapsed_seconds

In [ ]:
# warm up
print("Warming up regular Python UDF...")
_ = benchmark_dataframe(udf_df, "UDF warm-up")

print("\nWarming up built-in Spark expression...")
_ = benchmark_dataframe(builtin_df, "Built-in warm-up")

print("\nWarm-up completed.")

Warming up regular Python UDF...
UDF warm-up: 3.524 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]

Warming up built-in Spark expression...
Built-in warm-up: 0.346 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]

Warm-up completed.


In [ ]:
# running 5 benchmark rounds
udf_times = []
builtin_times = []

for round_num in range(1, 6):
    print(f"\n========== Round {round_num} ==========")

    if round_num % 2 == 1:
        udf_times.append(
            benchmark_dataframe(udf_df, f"UDF Round {round_num}")
        )
        builtin_times.append(
            benchmark_dataframe(builtin_df, f"Built-in Round {round_num}")
        )
    else:
        builtin_times.append(
            benchmark_dataframe(builtin_df, f"Built-in Round {round_num}")
        )
        udf_times.append(
            benchmark_dataframe(udf_df, f"UDF Round {round_num}")
        )

print("\nAll benchmark rounds completed.")


========== Round 1 ==========
UDF Round 1: 3.701 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]
Built-in Round 1: 0.450 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]

========== Round 2 ==========
Built-in Round 2: 0.298 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]
UDF Round 2: 2.578 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]

========== Round 3 ==========
UDF Round 3: 2.283 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_status='MEDIUM', count=400000), Row(amount_status='HIGH', count=300000)]
Built-in Round 3: 0.527 seconds
Result: [Row(amount_status='LOW', count=300000), Row(amount_statu

In [ ]:
# final summary
import statistics

udf_median = statistics.median(udf_times)
builtin_median = statistics.median(builtin_times)

print("UDF timings:", [round(t, 3) for t in udf_times])
print("Built-in timings:", [round(t, 3) for t in builtin_times])

print(f"\nUDF median: {udf_median:.3f} seconds")
print(f"Built-in median: {builtin_median:.3f} seconds")



UDF timings: [3.701, 2.578, 2.283, 2.304, 3.794]
Built-in timings: [0.45, 0.298, 0.527, 0.291, 0.458]

UDF median: 2.578 seconds
Built-in median: 0.450 seconds


In [ ]:
from IPython.display import display, Markdown

speedup = udf_median / builtin_median if builtin_median > 0 else None

if speedup is not None and speedup > 1:
    conclusion = (
        f"The built-in Spark expression was {speedup:.2f}x faster "
        "than the regular Python UDF in this benchmark."
    )
elif speedup is not None and speedup < 1:
    conclusion = (
        f"The regular Python UDF was {1 / speedup:.2f}x faster "
        "in this benchmark. More investigation is needed."
    )
else:
    conclusion = (
        "The implementations had equal median runtimes, or the "
        "speedup could not be calculated."
    )

report = f"""
# Spark Performance Optimization — Findings Report

## 1. Objective
Compare a regular Python UDF with an equivalent built-in Spark expression
for classifying one million records.

## 2. Environment and approach
- Framework: PySpark {spark.version}
- Dataset: {base_df.count():,} generated rows
- Execution mode: local[*]
- Adaptive Query Execution: {spark.conf.get("spark.sql.adaptive.enabled")}
- Before optimization: regular Python UDF
- After optimization: built-in when() expression
- Measurement: warm-up followed by five alternating runs
- Metric: median elapsed time for grouping and counting results

## 3. Benchmark results

| Metric | Regular Python UDF | Built-in expression |
|---|---:|---:|
| Median runtime (seconds) | {udf_median:.3f} | {builtin_median:.3f} |

Speedup ratio (UDF median / built-in median):
{speedup:.2f}x
if speedup is not None else "Not available"

## 4. Findings
{conclusion}

The execution plans were inspected using explain("formatted").
The benchmark includes aggregation and result collection, so it measures
the complete tested action, not only expression evaluation.

## 5. Troubleshooting approach
- Slow outlier task: inspect task duration and partition sizes for skew.
- High shuffle: inspect joins, groupBy, distinct, and repartition operations.
- Python worker or serialization failure: inspect UDF logic and Python/JVM setup.
- Driver memory pressure: avoid collecting large datasets to the driver.
- Slow UDF: test an equivalent built-in expression and benchmark it.


## 6. Conclusion
Use built-in Spark expressions when they can express the required logic
correctly. Validate output equivalence and measure performance before
adopting an optimization.
"""

display(Markdown(report))


# Spark Performance Optimization — Findings Report

## 1. Objective
Compare a regular Python UDF with an equivalent built-in Spark expression
for classifying one million records.

## 2. Environment and approach
- Framework: PySpark 4.2.0
- Dataset: 1,000,000 generated rows
- Execution mode: local[*]
- Adaptive Query Execution: true
- Before optimization: regular Python UDF
- After optimization: built-in when() expression
- Measurement: warm-up followed by five alternating runs
- Metric: median elapsed time for grouping and counting results

## 3. Benchmark results

| Metric | Regular Python UDF | Built-in expression |
|---|---:|---:|
| Median runtime (seconds) | 2.578 | 0.450 |

Speedup ratio (UDF median / built-in median):
5.73x
if speedup is not None else "Not available"

## 4. Findings
The built-in Spark expression was 5.73x faster than the regular Python UDF in this benchmark.

The execution plans were inspected using explain("formatted").
The benchmark includes aggregation and result collection, so it measures
the complete tested action, not only expression evaluation.

## 5. Troubleshooting approach
- Slow outlier task: inspect task duration and partition sizes for skew.
- High shuffle: inspect joins, groupBy, distinct, and repartition operations.
- Python worker or serialization failure: inspect UDF logic and Python/JVM setup.
- Driver memory pressure: avoid collecting large datasets to the driver.
- Slow UDF: test an equivalent built-in expression and benchmark it.


## 6. Conclusion
Use built-in Spark expressions when they can express the required logic
correctly. Validate output equivalence and measure performance before
adopting an optimization.
